In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

def build_gold_indicadores_risco():
    print("Construindo a gold_indicadores_risco...")
    
    query = """
    CREATE OR REPLACE TABLE workspace.default.gold_indicadores_risco AS
    
    SELECT 
        c.id_cliente,
        c.nome,
        c.segmento,
        
        -- Métricas de Risco e Fraude
        COUNT(DISTINCT e.id_evento) AS qtd_eventos_risco,
        SUM(CASE WHEN e.severidade = 'Alta' THEN 1 ELSE 0 END) AS qtd_eventos_alta_severidade,
        
        -- Métricas de Estorno e Chargeback
        COUNT(DISTINCT est.id_estorno) AS qtd_estornos,
        SUM(CASE WHEN est.motivo = 'Chargeback' THEN 1 ELSE 0 END) AS qtd_chargebacks,
        
        -- Exposição Financeira (Total de valor envolvido em problemas)
        SUM(f.valor_bruto) AS valor_total_exposto_risco
        
    FROM workspace.default.gold_fato_transacao f
    
    -- Navegação Fato -> Dimensões
    JOIN workspace.default.silver_cartoes cart 
        ON f.id_cartao = cart.id_cartao
    JOIN workspace.default.silver_contas cont 
        ON cart.id_conta = cont.id_conta
    JOIN workspace.default.silver_clientes c 
        ON cont.id_cliente = c.id_cliente
        
    -- Joins com as tabelas de problemas (LEFT JOIN para manter a rastreabilidade)
    LEFT JOIN workspace.default.silver_eventos_risco e 
        ON f.id_transacao = e.id_transacao
    LEFT JOIN workspace.default.silver_estornos est 
        ON f.id_transacao = est.id_transacao
        
    -- Filtra apenas as transações que tiveram algum BO
    WHERE e.id_evento IS NOT NULL OR est.id_estorno IS NOT NULL
    
    GROUP BY 1, 2, 3;
    """
    
    spark.sql(query)
    print("Sucesso! Tabela workspace.default.gold_indicadores_risco criada.")

build_gold_indicadores_risco()